<a href="https://colab.research.google.com/github/henrique-furtado47/Sistema-de-caixa---REMAKE/blob/main/Sistema_de_caixa_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Caixa feito em Python — POO
## Desenvolvido por Henrique Furtado

Remake do sistema de caixa com estoque e carrinho, agora utilizando **Programação Orientada a Objetos**.

**Bibliotecas:** Pandas, IPython.display

In [2]:
from __future__ import annotations
!pip install pandas
import pandas as pd
from IPython.display import clear_output, display

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 3.1 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 3.6 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/home/henrique.furtado/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


ModuleNotFoundError: No module named 'pandas'

In [ ]:
class Produto:
    """Representa um produto com nome, quantidade, preço e código."""

    _proximo_codigo: int = 1  # auto-incremento compartilhado

    def __init__(self, nome: str, quantidade: int, preco: float, codigo: int | None = None) -> None:
        self.nome = nome
        self.quantidade = quantidade
        self.preco = preco
        if codigo is not None:
            self.codigo = codigo
        else:
            self.codigo = Produto._proximo_codigo
            Produto._proximo_codigo += 1

    def __repr__(self) -> str:
        return f"Produto({self.nome!r}, qtd={self.quantidade}, R${self.preco:.2f}, cod={self.codigo})"

In [ ]:
class Estoque:
    """Gerencia a coleção de produtos disponíveis."""

    def __init__(self) -> None:
        self._produtos: list[Produto] = []

    # ---------- helpers ----------
    def _buscar_por_codigo(self, codigo: int) -> Produto | None:
        for p in self._produtos:
            if p.codigo == codigo:
                return p
        return None

    def _buscar_por_nome(self, nome: str) -> Produto | None:
        for p in self._produtos:
            if p.nome.lower() == nome.lower():
                return p
        return None

    # ---------- CRUD ----------
    def cadastrar(self) -> None:
        nome = input("Digite o nome do produto: ")
        quantidade = int(input("Digite a quantidade: "))
        preco = float(input("Digite o preço: "))
        produto = Produto(nome, quantidade, preco)
        self._produtos.append(produto)
        print(f"Produto '{produto.nome}' cadastrado com código {produto.codigo}.")

    def remover(self) -> None:
        codigo = int(input("Digite o código do produto: "))
        produto = self._buscar_por_codigo(codigo)
        if produto is None:
            print("Produto não encontrado.")
            return
        self._produtos.remove(produto)
        print("Produto removido com sucesso!")

    def atualizar(self) -> None:
        codigo = int(input("Digite o código do produto: "))
        produto = self._buscar_por_codigo(codigo)
        if produto is None:
            print("Produto não encontrado.")
            return

        print("1 - Atualizar nome")
        print("2 - Atualizar quantidade")
        print("3 - Atualizar preço")
        print("0 - Cancelar")
        opcao = int(input("Opção: "))

        if opcao == 1:
            produto.nome = input("Novo nome: ")
        elif opcao == 2:
            produto.quantidade = int(input("Nova quantidade: "))
        elif opcao == 3:
            produto.preco = float(input("Novo preço: "))
        elif opcao == 0:
            return
        else:
            print("Opção inválida!")
            return
        print("Produto atualizado com sucesso!")

    def localizar(self) -> None:
        print("1 - Localizar por nome\n2 - Localizar por código")
        opcao = int(input("Opção: "))
        if opcao == 1:
            nome = input("Nome do produto: ")
            produto = self._buscar_por_nome(nome)
        elif opcao == 2:
            codigo = int(input("Código do produto: "))
            produto = self._buscar_por_codigo(codigo)
        else:
            print("Opção inválida!")
            return

        if produto is None:
            print("Produto não encontrado.")
        else:
            print(f"Encontrado: {produto}")

    def visualizar(self) -> None:
        clear_output()
        if not self._produtos:
            print("Estoque vazio.")
            return
        df = pd.DataFrame([
            {"Código": p.codigo, "Nome": p.nome, "Quantidade": p.quantidade, "Preço": p.preco}
            for p in self._produtos
        ])
        display(df)

    def obter_produto(self, codigo: int) -> Produto | None:
        return self._buscar_por_codigo(codigo)

In [ ]:
class ItemCarrinho:
    """Representa um item adicionado ao carrinho (referência ao produto + quantidade)."""

    def __init__(self, produto: Produto, quantidade: int) -> None:
        self.produto = produto
        self.quantidade = quantidade

    @property
    def subtotal(self) -> float:
        return self.produto.preco * self.quantidade

In [ ]:
class Carrinho:
    """Gerencia os itens selecionados para compra."""

    def __init__(self) -> None:
        self._itens: list[ItemCarrinho] = []

    # ---------- helpers ----------
    def _buscar_item(self, codigo: int) -> ItemCarrinho | None:
        for item in self._itens:
            if item.produto.codigo == codigo:
                return item
        return None

    @property
    def total(self) -> float:
        return sum(item.subtotal for item in self._itens)

    @property
    def vazio(self) -> bool:
        return len(self._itens) == 0

    # ---------- ações ----------
    def adicionar(self, estoque: Estoque) -> None:
        codigo = int(input("Código do produto: "))
        produto = estoque.obter_produto(codigo)
        if produto is None:
            print("Produto não encontrado no estoque!")
            return

        quantidade = int(input("Quantidade: "))
        if quantidade > produto.quantidade:
            print("Quantidade insuficiente no estoque!")
            return

        item_existente = self._buscar_item(codigo)
        if item_existente:
            item_existente.quantidade += quantidade
        else:
            self._itens.append(ItemCarrinho(produto, quantidade))
        print("Produto adicionado ao carrinho!")

    def remover(self) -> None:
        codigo = int(input("Código do produto: "))
        item = self._buscar_item(codigo)
        if item is None:
            print("Produto não encontrado no carrinho.")
            return

        quantidade = int(input("Quantidade a remover: "))
        if quantidade > item.quantidade:
            print("Quantidade maior do que a presente no carrinho!")
        elif quantidade == item.quantidade:
            self._itens.remove(item)
            print("Produto removido do carrinho!")
        else:
            item.quantidade -= quantidade
            print("Quantidade atualizada no carrinho!")

    def limpar(self) -> None:
        opcao = input("Tem certeza? (s/n): ").strip().lower()
        if opcao == "s":
            self._itens.clear()
            clear_output()
            print("Carrinho limpo!")
        else:
            print("Operação cancelada.")

    def visualizar(self) -> None:
        clear_output()
        if self.vazio:
            print("Carrinho vazio.")
            return
        df = pd.DataFrame([
            {
                "Código": i.produto.codigo,
                "Nome": i.produto.nome,
                "Qtd": i.quantidade,
                "Preço Unit.": i.produto.preco,
                "Subtotal": i.subtotal,
            }
            for i in self._itens
        ])
        display(df)
        print(f"\nTotal: R${self.total:.2f}")

    def finalizar(self) -> list[ItemCarrinho]:
        """Retorna os itens e esvazia o carrinho."""
        itens = list(self._itens)
        self._itens.clear()
        return itens

In [ ]:
class Pagamento:
    """Calcula valores de pagamento (à vista, parcelado, troco)."""

    PARCELAS_SEM_JUROS = 6

    @staticmethod
    def calcular_parcela(total: float, parcelas: int) -> float:
        if parcelas <= Pagamento.PARCELAS_SEM_JUROS:
            return total / parcelas
        juros = parcelas / 150  # taxa proporcional
        return total * (1 + juros) / parcelas

    @staticmethod
    def processar(total: float) -> None:
        print("Método de pagamento:")
        print("1 - Cartão de crédito")
        print("2 - Cartão de débito")
        print("3 - Dinheiro")
        opcao = int(input("Opção: "))

        if opcao == 1:
            parcelas = int(input("Número de parcelas: "))
            valor_parcela = Pagamento.calcular_parcela(total, parcelas)
            print(f"{parcelas}x de R${valor_parcela:.2f}")
        elif opcao == 2:
            print(f"Total no débito: R${total:.2f}")
        elif opcao == 3:
            dinheiro = float(input("Valor recebido: "))
            troco = dinheiro - total
            if troco < 0:
                print("Valor insuficiente!")
                return
            print(f"Troco: R${troco:.2f}")
        else:
            print("Opção inválida!")
            return

        print("Pagamento realizado com sucesso!")

In [ ]:
class SistemaDeCaixa:
    """Orquestra estoque, carrinho e pagamento."""

    def __init__(self) -> None:
        self.estoque = Estoque()
        self.carrinho = Carrinho()

    # ---------- menus ----------
    def _menu_estoque(self) -> None:
        opcoes = {
            1: self.estoque.cadastrar,
            2: self.estoque.visualizar,
            3: self.estoque.remover,
            4: self.estoque.atualizar,
            5: self.estoque.localizar,
        }
        print("\n--- Estoque ---")
        print("1 - Cadastrar produto")
        print("2 - Listar produtos")
        print("3 - Remover produto")
        print("4 - Atualizar produto")
        print("5 - Localizar produto")
        print("0 - Voltar")
        opcao = int(input("Opção: "))
        acao = opcoes.get(opcao)
        if acao:
            acao()
        elif opcao != 0:
            print("Opção inválida!")

    def _menu_carrinho(self) -> None:
        print("\n--- Carrinho ---")
        print("1 - Adicionar produto")
        print("2 - Listar produtos")
        print("3 - Limpar carrinho")
        print("4 - Remover produto")
        print("5 - Finalizar compra")
        print("0 - Voltar")
        opcao = int(input("Opção: "))

        if opcao == 1:
            self.carrinho.adicionar(self.estoque)
        elif opcao == 2:
            self.carrinho.visualizar()
        elif opcao == 3:
            self.carrinho.limpar()
        elif opcao == 4:
            self.carrinho.remover()
        elif opcao == 5:
            self._finalizar_compra()
        elif opcao == 0:
            clear_output()
        else:
            print("Opção inválida!")

    def _finalizar_compra(self) -> None:
        if self.carrinho.vazio:
            print("Carrinho está vazio!")
            return

        total = self.carrinho.total
        itens = self.carrinho.finalizar()

        # Atualiza o estoque
        for item in itens:
            item.produto.quantidade -= item.quantidade

        clear_output()
        print(f"Total da compra: R${total:.2f}")
        Pagamento.processar(total)
        print("Compra finalizada com sucesso!")

    # ---------- loop principal ----------
    def executar(self) -> None:
        while True:
            print("\n=== Sistema de Caixa ===")
            print("1 - Acessar estoque")
            print("2 - Acessar carrinho")
            print("0 - Sair")
            opcao = int(input("Opção: "))

            if opcao == 1:
                self._menu_estoque()
            elif opcao == 2:
                self._menu_carrinho()
            elif opcao == 0:
                clear_output()
                print("FIM DO PROGRAMA")
                break
            else:
                print("Opção inválida!")

In [ ]:
# Inicialização e execução do sistema
sistema = SistemaDeCaixa()
sistema.executar()

NameError: name 'SistemaDeCaixa' is not defined